# Traffic Congestion Prediction: MobileNet + Echo State Network (ESN)

This notebook implements a **training-free** pipeline for video classification, suitable for cloud environments (Kaggle/Colab).

### Approach (Option 4):
1.  **Spatial Features**: Use a **Frozen Pre-trained MobileNetV2** to extract high-level visual features (1280-dim) from each frame. No backpropagation is performed on the CNN.
2.  **Temporal Features**: Use an **Echo State Network (ESN)** (Reservoir Computing) to model the temporal evolution of these features over time.
3.  **Classification**: Train a linear readout (Ridge Regression) to predict traffic congestion levels.

**Advantages**: 
- **Extremely Fast Training**: No gradient descent, just one-shot matrix solution.
- **Low Compute**: Can perform reasonably well without high-end GPUs for training.
- **Video-Native**: Processes raw video frames.

In [ ]:
# Install dependencies
!pip install opencv-python-headless scikit-learn polars

## 1. Configuration & Paths
Please set the following paths to your dataset locations.

In [ ]:
# --- USER CONFIGURATION ---
BASE_DIR = '/teamspace/studios/this_studio/Barbados_Traffic_Analysis_Challenge_dev' # Example
VIDEO_DIR = '/teamspace/studios/this_studio/videos' # Example
TRAIN_CSV = os.path.join(BASE_DIR, 'demos/Train.csv')
TEST_CSV = os.path.join(BASE_DIR, 'demos/TestInputSegments.csv')
SAMPLE_SUB = os.path.join(BASE_DIR, 'demos/SampleSubmission.csv')

# Model Config
IMG_SIZE = (224, 224)
SEQ_LENGTH = 30         # Number of frames to extract per video (downsampled)
RESERVOIR_DIM = 1000    # ESN Reservoir Size
SPECTRAL_RADIUS = 0.9
LEAK_RATE = 0.2
RIDGE_ALPHA = 1.0
# --------------------------

In [ ]:
import os
from google.oauth2 import service_account

# If you uploaded the file to a dataset:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/teamspace/studios/this_studio/tokens.json"


In [ ]:
client = storage.Client(project="brb-traffic")

# Base directories and bucket name
bucket_name = 'brb-traffic'

# Video paths
video_dir = VIDEO_DIR
video_path = '/teamspace/studios/this_studio/videos'
os.makedirs(video_dir, exist_ok=True)

# Datasheet paths
train_csv_path = TRAIN_CSV
sample_submission_csv_path = SAMPLE_SUB

In [ ]:
# Load the dataset
train = pd.read_csv(TRAIN_CSV)

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

In [ ]:
ss = pd.read_csv(SAMPLE_SUB)
display(ss.shape,ss.head())

In [ ]:
# %%capture
blobs=train.videos.tolist()[0:2000]
print(f"Number of blobs selected: {len(blobs)}")
display(blobs)

In [ ]:
# %%capture
from google.api_core.exceptions import NotFound

print(f"--- Debugging Blobs Variable ---")
print(f"Type of 'blobs' before loop: {type(blobs)}")
print(f"Content of 'blobs' (first 5): {blobs[:5]}")
print(f"----------------------------------")

# --- Existing loop for downloading files from the 'blobs' list ---
for blob_name in blobs:
    # Ensure blob_name is a string before proceeding
    if not isinstance(blob_name, str):
        print(f"❌ Error: Expected string for blob_name, but got {type(blob_name)}. Skipping.")
        continue

    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print(f"Attempting to download blob: '{blob_name}' to '{local_path}'") # Added print for clarity
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading '{blob_name}': {e}")

## 2. Feature Extraction (MobileNetV2)
We load a MobileNetV2 pre-trained on ImageNet, remove the top classification layer, and use global average pooling to get a 1280-dimensional vector for each frame.

In [ ]:
def build_feature_extractor():
    base_model = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        pooling='avg',
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Freeze weights
    return base_model

feat_extractor = build_feature_extractor()
print("Feature Extractor Loaded: MobileNetV2 (Frozen)")

## 3. Video Processing Utils
Functions to read video frames and extract features.

In [ ]:
def get_video_features(video_path):
    frames = extract_frames(video_path)
    if len(frames) == 0:
        return np.zeros(1280) # Return zero vector if empty
    
    # Extract frame-level features: (T_frames, 1280)
    frame_features = feat_extractor.predict(frames, verbose=0)
    
    # Temporal Pooling: Average frame features to get one vector per video
    # Shape: (1280,)
    video_feature = np.mean(frame_features, axis=0)
    return video_feature

## 4. Echo State Network Class
A pure NumPy implementation of ESN for sequence classification.

In [ ]:
def create_sequential_blocks(df, is_test=False):
    """
    Groups a dataframe into sequential blocks of videos based on time_segment_id.
    Returns: List of (X_block, y_block) tuples.
    X_block: (T_videos, 1280)
    y_block: (T_videos, )
    """
    blocks = []
    
    # Ensure sorted by view/camera and time
    # Assuming 'time_segment_id' implies order. If datetime is available, sort by it.
    if 'datetimestamp_start' in df.columns:
        df['dt'] = pd.to_datetime(df['datetimestamp_start'])
        df = df.sort_values(by=['view_label', 'dt'])
    else:
        df = df.sort_values(by=['view_label', 'time_segment_id'])
    
    print(f"Grouping {len(df)} samples into sequential blocks...")
    
    grouped = df.groupby('view_label')
    
    for view_id, group in grouped:
        # Identify continuity breaks
        # We assume time_segment_id is continuous integers (0, 1, 2...) for continuous time
        # diff != 1 implies a break in the sequence
        group = group.copy()
        group['block_id'] = (group['time_segment_id'].diff() != 1).cumsum()
        
        for _, block in group.groupby('block_id'):
            # Stack features
            if 'features' not in block.columns:
                continue 
                
            X_seq = np.stack(block['features'].values)
            
            if not is_test:
                y_seq = block['label_code'].values
            else:
                y_seq = np.zeros(len(block)) # Dummy labels for test
                
            blocks.append((X_seq, y_seq, block['video_full_path'].values if is_test else None))
            
    print(f"Refined into {len(blocks)} blocks.")
    return blocks

In [ ]:
class SequentialESN:
    """ESN that processes sequences of videos (inter-video modeling)."""
    def __init__(self, input_dim=1280, res_dim=2000, rho=0.9, leak=0.2, alpha=1.0):
        self.res_dim = res_dim
        self.rho = rho
        self.leak = leak
        self.alpha = alpha
        
        rng = np.random.RandomState(42)
        self.W_in = rng.uniform(-1, 1, (res_dim, input_dim))
        self.W_res = rng.uniform(-1, 1, (res_dim, res_dim))
        mask = rng.rand(res_dim, res_dim) > 0.95
        self.W_res[mask] = 0
        
        try:
            eigenvalues = np.linalg.eigvals(self.W_res)
            max_eig = np.max(np.abs(eigenvalues))
            self.W_res *= (self.rho / max_eig)
        except:
            self.W_res *= 0.9
        
        self.readout = Ridge(alpha=self.alpha)
        
    def get_states_sequence(self, input_seq):
        T = input_seq.shape[0]
        states = np.zeros((T, self.res_dim))
        x = np.zeros(self.res_dim)
        
        for t in range(T):
            u = input_seq[t]
            pre = np.dot(self.W_in, u) + np.dot(self.W_res, x)
            x = (1 - self.leak) * x + self.leak * np.tanh(pre)
            states[t] = x
        return states
    
    def fit(self, blocks, compute_metrics=True):
        all_states = []
        all_targets = []
        
        print(f"Training ESN on {len(blocks)} blocks...")
        for X_seq, y_seq, _ in blocks:
            states_seq = self.get_states_sequence(X_seq)
            all_states.append(states_seq)
            all_targets.append(y_seq)
            
        X_train_res = np.vstack(all_states)
        y_train_flat = np.concatenate(all_targets)
        
        self.readout.fit(X_train_res, y_train_flat)
        
        if compute_metrics:
            y_pred = self.readout.predict(X_train_res)
            y_pred_class = np.round(np.clip(y_pred, 0, 3)).astype(int)
            
            acc = accuracy_score(y_train_flat, y_pred_class)
            f1 = f1_score(y_train_flat, y_pred_class, average='macro')
            print(f"[TRAIN] Accuracy: {acc:.4f} | F1-Macro: {f1:.4f}")
        
    def predict(self, blocks):
        all_preds = []
        for X_seq, _, _ in blocks:
            states_seq = self.get_states_sequence(X_seq)
            preds_seq = self.readout.predict(states_seq)
            all_preds.append(preds_seq)
        return all_preds

In [ ]:
def reconstruct_test_video_path(row):
    # Example Input columns: 'view_label' ('Norman Niles #1'), 'datetimestamp_start' ('2025-10-20 06:00:45')
    # Target Format: 'normanniles1/normanniles1_2025-10-20-06-00-45.mp4'
    # Or simply joining with VIDEO_DIR if folders are flattened: 'normanniles1_2025-10-20-06-00-45.mp4'
    
    # 1. Normalize Camera Name
    cam_map = {
        'Norman Niles #1': 'normanniles1',
        'Norman Niles #2': 'normanniles2',
        'Norman Niles #3': 'normanniles3',
        'Norman Niles #4': 'normanniles4'
    }
    
    cam_key = row.get('view_label', '')
    if cam_key not in cam_map:
        # Fallback if view_label is missing but implied by ID
        return None
        
    cam_id = cam_map[cam_key]
    
    # 2. Extract Date/Time
    # Assuming 'datetimestamp_start' exists in Test CSV (Standard Zindi format)
    # If not, we might need to parse 'ID' (e.g. time_segment_0... usually doesn't have date)
    # BUT: The TestInputSegments.csv provided usually has 'video_time' or 'datetimestamp_start'
    
    dt_str = row.get('datetimestamp_start', row.get('video_time', ''))
    if not dt_str:
        return None
        
    # Format: '2025-10-20 06:00:45' -> '2025-10-20-06-00-45'
    dt_formatted = str(dt_str).replace(' ', '-').replace(':', '-')
    
    # 3. Construct Path
    # Assuming folder structure 'normanniles1/filename.mp4' to match training
    filename = f"{cam_id}_{dt_formatted}.mp4"
    full_path = os.path.join(VIDEO_DIR, cam_id, filename)
    
    # Check if we need to fall back to flat directory
    if not os.path.exists(full_path):
         # Try flat in VIDEO_DIR
         full_path_flat = os.path.join(VIDEO_DIR, filename)
         # Return flat path anyway to let the loader check existence
         return full_path_flat
         
    return full_path

## 5. Load Data & Train
We will load video paths, labels, and run the pipeline.

In [ ]:
# --- 5. Inference (Test Blocks) ---

df_test = pd.read_csv(TEST_CSV)
print(f"Test Set Size: {len(df_test)}")

# Apply Path Reconstruction
print("Reconstructing Test Video Paths...")
df_test['video_full_path'] = df_test.apply(reconstruct_test_video_path, axis=1)

# Extract features
print("Extracting Test Features...")
test_feats = []
missing_count = 0
total_test = len(df_test)

for idx, row in df_test.iterrows():
    if idx % 100 == 0: print(f"{idx}/{total_test}")
    path = row['video_full_path']
    if path and os.path.exists(path):
        test_feats.append(get_video_features(path))
    else:
        # Handle missing video (Zero vector or mean)
        # print(f"Missing: {path}")
        test_feats.append(np.zeros(1280))
        missing_count += 1
        
print(f"Test Extraction Complete. Missing Videos: {missing_count}/{total_test}")
df_test['features'] = test_feats

# Validation Metrics were calculated in Step 4 above.
print("Validation Metrics (Recap):")
print(f"Validation F1-Macro: {f1_score(y_val_true, y_val_pred_class, average='macro'):.4f}")

# Helper to fill submission from predictions
test_blocks = create_sequential_blocks(df_test, is_test=True)
test_preds_list = esn.predict(test_blocks)

# Map predictions back to DataFrame order is tricky with blocks.
# Simplified: Since we didn't drop rows in create_sequential_blocks (just grouped),
# we can leverage the sorted order.

# Better Strategy: create_sequential_blocks should return indices or we iteratively map.
# For now, we will create a dictionary map: {video_path: prediction}
# NOTE: This assumes video_path is unique per segment.

pred_map = {}
for i, p_seq in enumerate(test_preds_list):
    _, _, paths_seq = test_blocks[i]
    for j, path in enumerate(paths_seq):
        pred_val = np.round(np.clip(p_seq[j], 0, 3)).astype(int)
        if path is not None:
            pred_map[path] = pred_val

# Map back to df_test
df_test['pred_code'] = df_test['video_full_path'].map(pred_map).fillna(0).astype(int)

# Create Submission
congestion_inv_map = {0: 'free flowing', 1: 'light delay', 2: 'moderate delay', 3: 'heavy delay'}
submission = df_test[['ID']].copy()
# Target column name usually 'congestion_enter_rating' or just 'Target' depending on competition
submission['Target'] = df_test['pred_code'].map(congestion_inv_map)

submission.to_csv('submission_esn.csv', index=False)
print("Submission saved to submission_esn.csv")
print(submission.head())

## 6. Inference on Test Set
This section generates the submission file.